# VulGCL — Aliyun DSW (Full Pipeline)
**Phase 1 → Phase 2 → Train 4 models**

**Before running:**
1. Upload the entire `VulGCL` project folder to `/mnt/workspace/VulGCL`
2. If you mounted OSS, change `SAVE_DIR` in Cell 3 to your OSS path so results persist
3. Run cells top to bottom — each cell resumes where it left off (skip-if-exists)

In [ ]:
# ── Cell 1: Install missing packages ─────────────────────────────────────────
# The Aliyun DSW image already has: PyTorch 2.10, CUDA 12.8, numpy, tqdm, scikit-learn
# Only install what's not in the image:
!pip install transformers torch-geometric torch-scatter torch-sparse networkx 
print("Done")

In [ ]:
# ── Cell 2: Check Java + install Joern ────────────────────────────────────────
import os, subprocess

# Check Java (Joern needs Java 11+)
r = subprocess.run(["java", "-version"], capture_output=True, text=True)
if r.returncode == 0:
    print("Java:", r.stderr.strip().split('\n')[0])
else:
    print("Installing Java...")
    os.system("apt-get install -y default-jdk-headless -q")

# Install Joern v2.0.406 if not already present
JOERN_DIR = "/mnt/workspace/joern"
JOERN_BIN = f"{JOERN_DIR}/joern-cli"
if not os.path.exists(f"{JOERN_BIN}/joern"):
    print("Installing Joern v2.0.406 (takes ~2 min)...")
    os.makedirs(JOERN_DIR, exist_ok=True)
    os.system("curl -sL https://github.com/joernio/joern/releases/download/v2.0.406/joern-install.sh -o /tmp/joern-install.sh")
    os.system(f"chmod +x /tmp/joern-install.sh && /tmp/joern-install.sh --install-dir {JOERN_DIR} --yes")
    print("Joern installed.")
else:
    print("Joern already installed.")

os.environ["PATH"] = os.environ["PATH"] + f":{JOERN_BIN}"
r = subprocess.run(["which", "joern"], capture_output=True, text=True)
print("joern:", r.stdout.strip() or "NOT FOUND — check install")

In [ ]:
# ── Cell 3: Setup paths ───────────────────────────────────────────────────────
import os, glob, torch

PROJECT_DIR  = "/mnt/workspace/VulGCL"
os.chdir(PROJECT_DIR)

PKL_ROOT     = f"{PROJECT_DIR}/data/devign/processed/graphs_nx"
SAVE_DIR     = "/mnt/data"
PT_ROOT      = f"{SAVE_DIR}/pt_files"
WORK_DIR     = SAVE_DIR
CODEBERT_DIR = "/mnt/workspace/models/codebert"

for split in ["train", "validation", "test"]:
    os.makedirs(f"{PT_ROOT}/{split}", exist_ok=True)
os.makedirs(f"{WORK_DIR}/experiments/checkpoints", exist_ok=True)

assert os.path.isdir(CODEBERT_DIR), f"CodeBERT not found at {CODEBERT_DIR}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"CodeBERT : {CODEBERT_DIR}")
print(f"PKL root : {PKL_ROOT}")
print(f"PT root  : {PT_ROOT}  (OSS)")
print(f"Device   : {DEVICE}")

In [ ]:
# ── Cell 4: Phase 1 — Joern PDG extraction (CPU, ~3-4 hrs on 28 workers) ─────
# Resumes automatically — already-processed functions are skipped.
# Uses 28 of the 32 vCPU (leaves headroom for the OS).
!python src/data/preprocess.py --phase 1 --workers 28

In [ ]:
# ── Cell 5: Phase 2 functions (CodeBERT embedding) ────────────────────────────
import pickle
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

IMAGE_SIZE = 100
_CB_TOKENIZER = None
_CB_MODEL = None

def _get_codebert(device):
    global _CB_TOKENIZER, _CB_MODEL
    if _CB_TOKENIZER is None:
        print("Loading CodeBERT...")
        _CB_TOKENIZER = AutoTokenizer.from_pretrained(CODEBERT_DIR, local_files_only=True)
        _CB_MODEL = AutoModel.from_pretrained(CODEBERT_DIR, local_files_only=True)
        _CB_MODEL.eval()
    return _CB_TOKENIZER, _CB_MODEL.to(device)

def _embed_codes(codes, device, batch_size=64):
    tok, model = _get_codebert(device)
    vecs = []
    with torch.no_grad():
        for i in range(0, len(codes), batch_size):
            enc = tok(codes[i:i+batch_size], return_tensors="pt",
                      max_length=128, truncation=True, padding="max_length")
            out = model(input_ids=enc["input_ids"].to(device),
                        attention_mask=enc["attention_mask"].to(device))
            vecs.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(vecs, dim=0)

def pdg_to_pyg(G, label, device):
    nodes = list(G.nodes())
    if not nodes:
        return Data(x=torch.zeros(1, 768),
                    edge_index=torch.zeros(2, 0, dtype=torch.long),
                    y=torch.tensor([label], dtype=torch.float))
    x = _embed_codes([G.nodes[n].get("code", "") for n in nodes], device)
    n2i = {n: i for i, n in enumerate(nodes)}
    edges = list(G.edges())
    if edges:
        edge_index = torch.tensor([[n2i[s] for s, _ in edges],
                                   [n2i[d] for _, d in edges]], dtype=torch.long)
    else:
        edge_index = torch.zeros(2, 0, dtype=torch.long)
    return Data(x=x, edge_index=edge_index,
                y=torch.tensor([label], dtype=torch.float))

def pdg_to_image(G, x_emb):
    n = G.number_of_nodes()
    if n == 0:
        return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
    nodes = list(G.nodes())
    emb = x_emb[:n].float().numpy()
    deg   = nx.degree_centrality(G)
    close = nx.closeness_centrality(G)
    try:
        katz = nx.katz_centrality(G, alpha=0.01, max_iter=1000)
    except Exception:
        katz = {nd: 0.0 for nd in nodes}
    channels = []
    for c_dict in [deg, close, katz]:
        scores = np.array([c_dict.get(nd, 0.0) for nd in nodes],
                          dtype=np.float32).reshape(-1, 1)
        channels.append(scores * emb)
    mat = np.stack(channels, axis=0)
    for ch in range(3):
        mx = np.abs(mat[ch]).max()
        if mx > 0:
            mat[ch] /= mx
    t = torch.from_numpy(mat).unsqueeze(0)
    return F.interpolate(t, size=(IMAGE_SIZE, IMAGE_SIZE),
                         mode="bilinear", align_corners=False).squeeze(0)

def pdg_to_slice(G, top_k=10):
    if G.number_of_nodes() == 0:
        return ""
    cent = nx.betweenness_centrality(G)
    top  = sorted(cent, key=cent.get, reverse=True)[:top_k]
    top.sort(key=lambda nd: G.nodes[nd].get("line", 0))
    return " ".join(G.nodes[nd].get("code", "").strip()
                    for nd in top if G.nodes[nd].get("code", ""))

print("Phase 2 functions ready.")

In [ ]:
# ── Cell 6: Run Phase 2 — CodeBERT embed → .pt files (~2-3 hrs on L20) ───────
def run_phase2(pkl_root, pt_root, splits=("train", "validation", "test")):
    for split in splits:
        pkl_dir   = f"{pkl_root}/{split}"
        pt_dir    = f"{pt_root}/{split}"
        os.makedirs(pt_dir, exist_ok=True)
        pkl_files = sorted(glob.glob(f"{pkl_dir}/*.pkl"))
        if not pkl_files:
            print(f"[Phase 2] {split}: no pkl files at {pkl_dir} — run Phase 1 first")
            continue
        print(f"\n[Phase 2] {split} — {len(pkl_files)} graphs")
        ok = skip = err = 0
        for pkl_path in tqdm(pkl_files, desc=f"  Embed/{split}"):
            stem   = os.path.splitext(os.path.basename(pkl_path))[0]
            pt_out = f"{pt_dir}/{stem}.pt"
            if os.path.exists(pt_out):
                skip += 1
                continue
            try:
                with open(pkl_path, "rb") as f:
                    obj = pickle.load(f)
                G, label  = obj["graph"], obj["label"]
                data      = pdg_to_pyg(G, label, DEVICE)
                img       = pdg_to_image(G, data.x)
                llm_slice = pdg_to_slice(G)
                torch.save({"graph": data, "image": img,
                            "llm_slice": llm_slice, "label": float(label)}, pt_out)
                ok += 1
            except Exception as e:
                err += 1
                tqdm.write(f"  error {stem}: {e}")
        print(f"  ok={ok}  skip={skip}  err={err}")

run_phase2(PKL_ROOT, PT_ROOT)
print("\nPhase 2 complete.")

In [ ]:
# ── Cell 6b: Reprocess GRAPH branch → structural features (no CodeBERT) ───────
# Decorrelates the graph branch from the LLM branch: instead of frozen CodeBERT
# node embeddings, the graph now carries node TYPE + STRUCTURE, learned end-to-end.
# Only the "graph" field is rebuilt; "image" and "llm_slice" are kept as-is.
import hashlib

NUM_TYPE_BUCKETS = 64   # must match GraphBranch in Cell 8

def _type_id(t):
    return int(hashlib.md5(str(t).encode()).hexdigest(), 16) % NUM_TYPE_BUCKETS

def pdg_to_pyg_struct(G, label):
    """Node features = [in_deg, out_deg, degree_centrality, rel_line] + type id.
    Fully structural — no CodeBERT, so the branch is orthogonal to the LLM branch."""
    nodes = list(G.nodes())
    if not nodes:
        return Data(x=torch.zeros(1, 4),
                    node_type=torch.zeros(1, dtype=torch.long),
                    edge_index=torch.zeros(2, 0, dtype=torch.long),
                    y=torch.tensor([label], dtype=torch.float))
    n2i   = {n: i for i, n in enumerate(nodes)}
    indeg = dict(G.in_degree())
    outdeg = dict(G.out_degree())
    degc  = nx.degree_centrality(G)
    lines = [(G.nodes[n].get("line", 0) or 0) for n in nodes]
    maxline = max(lines) or 1
    x = torch.tensor(
        [[float(indeg.get(n, 0)), float(outdeg.get(n, 0)),
          float(degc.get(n, 0.0)), (G.nodes[n].get("line", 0) or 0) / maxline]
         for n in nodes], dtype=torch.float)
    node_type = torch.tensor(
        [_type_id(G.nodes[n].get("type", "UNK")) for n in nodes], dtype=torch.long)
    edges = list(G.edges())
    if edges:
        edge_index = torch.tensor([[n2i[s] for s, _ in edges],
                                   [n2i[d] for _, d in edges]], dtype=torch.long)
    else:
        edge_index = torch.zeros(2, 0, dtype=torch.long)
    return Data(x=x, node_type=node_type, edge_index=edge_index,
                y=torch.tensor([label], dtype=torch.float))

def reprocess_graphs(pkl_root, pt_root, splits=("train", "validation", "test")):
    for split in splits:
        pkl_dir = f"{pkl_root}/{split}"
        pt_dir  = f"{pt_root}/{split}"
        pt_files = sorted(glob.glob(f"{pt_dir}/*.pt"))
        if not pt_files:
            print(f"[Reproc] {split}: no .pt files at {pt_dir}")
            continue
        ok = err = miss = 0
        for pt_path in tqdm(pt_files, desc=f"  Reproc/{split}"):
            stem = os.path.splitext(os.path.basename(pt_path))[0]
            pkl_path = f"{pkl_dir}/{stem}.pkl"
            if not os.path.exists(pkl_path):
                miss += 1
                continue
            try:
                obj = torch.load(pt_path, map_location="cpu", weights_only=False)
                with open(pkl_path, "rb") as f:
                    G = pickle.load(f)["graph"]
                obj["graph"] = pdg_to_pyg_struct(G, obj["label"])  # keep image + llm_slice
                torch.save(obj, pt_path)
                ok += 1
            except Exception as e:
                err += 1
                tqdm.write(f"  err {stem}: {e}")
        print(f"  {split}: ok={ok}  err={err}  miss={miss}")

reprocess_graphs(PKL_ROOT, PT_ROOT)
print("\nGraph reprocess complete — graph branch is now structural.")

In [ ]:
# ── Cell 7: Dataset + DataLoaders ─────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch
from transformers import AutoTokenizer

class VulDataset(Dataset):
    def __init__(self, split, pt_root, max_seq_len=512):
        self.tok         = AutoTokenizer.from_pretrained(CODEBERT_DIR, local_files_only=True)
        self.max_seq_len = max_seq_len
        self.pt_files    = sorted(glob.glob(f"{pt_root}/{split}/*.pt"))
        print(f"  {split}: {len(self.pt_files)} samples")

    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        obj  = torch.load(self.pt_files[idx], map_location="cpu", weights_only=False)
        text = obj.get("llm_slice", "") or ""
        enc  = self.tok(text, max_length=self.max_seq_len,
                        truncation=True, padding="max_length", return_tensors="pt")
        return {
            "graph":          obj["graph"],
            "image":          obj["image"],
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(obj["label"], dtype=torch.float),
        }

def collate_fn(samples):
    return {
        "graph":          Batch.from_data_list([s["graph"] for s in samples]),
        "image":          torch.stack([s["image"]          for s in samples]),
        "input_ids":      torch.stack([s["input_ids"]      for s in samples]),
        "attention_mask": torch.stack([s["attention_mask"] for s in samples]),
        "label":          torch.stack([s["label"]          for s in samples]),
    }

print("Loading datasets...")
train_ds = VulDataset("train",      PT_ROOT)
val_ds   = VulDataset("validation", PT_ROOT)
test_ds  = VulDataset("test",       PT_ROOT)

BS = 16
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  collate_fn=collate_fn, num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, collate_fn=collate_fn, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, collate_fn=collate_fn, num_workers=4)
print("DataLoaders ready.")

In [ ]:
# ── Cell 8: Models ────────────────────────────────────────────────────────────
import torch.nn as nn
from torch_geometric.nn import GATConv, global_add_pool
from torch_geometric.utils import softmax as geo_softmax
from transformers import RobertaModel

NUM_TYPE_BUCKETS = 64   # must match Cell 6b reprocess

class GraphBranch(nn.Module):
    """Structural GNN: learnable node-type embedding + degree/line features,
    trained end-to-end (no frozen CodeBERT). Attention pooling over nodes."""
    def __init__(self, struct_dim=4, type_emb=32, hidden_dim=256, num_layers=2):
        super().__init__()
        self.type_embedding = nn.Embedding(NUM_TYPE_BUCKETS, type_emb)
        in_dim = type_emb + struct_dim
        self.input_bn = nn.BatchNorm1d(in_dim)
        self.convs = nn.ModuleList([
            GATConv(in_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.relu = nn.ReLU()
        self.att  = nn.Linear(hidden_dim, 1)
    def forward(self, data):
        h = torch.cat([self.type_embedding(data.node_type), data.x], dim=-1)
        h = self.input_bn(h)
        for conv in self.convs:
            h = self.relu(conv(h, data.edge_index))
        w = geo_softmax(self.att(h), data.batch)      # attention weights per node
        return global_add_pool(h * w, data.batch)     # weighted sum → [B, hidden]

class ImageBranch(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Linear(128 * 4 * 4, hidden_dim)
    def forward(self, x):
        return self.fc(self.cnn(x).view(x.size(0), -1))

class LLMBranch(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained(CODEBERT_DIR, local_files_only=True)
        self.proj    = nn.Linear(768, hidden_dim)
    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.proj(cls)

class GraphClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = GraphBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        return self.head(self.branch(batch["graph"].to(DEVICE))).squeeze(-1)

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = ImageBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        return self.head(self.branch(batch["image"].to(DEVICE))).squeeze(-1)

class LLMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = LLMBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        return self.head(self.branch(
            batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)
        )).squeeze(-1)

class VulGCLModel(nn.Module):
    """Gated fusion of the three 256-d branch vectors + per-branch aux heads.
    forward() returns fused logits; forward_train() also returns the 3 aux logits."""
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.graph_branch = GraphBranch()
        self.image_branch = ImageBranch()
        self.llm_branch   = LLMBranch()
        self.gate = nn.Linear(3 * hidden_dim, 3)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1)
        )
        self.aux_g = nn.Linear(hidden_dim, 1)
        self.aux_i = nn.Linear(hidden_dim, 1)
        self.aux_l = nn.Linear(hidden_dim, 1)
    def _impl(self, batch):
        h_g = self.graph_branch(batch["graph"].to(DEVICE))
        h_i = self.image_branch(batch["image"].to(DEVICE))
        h_l = self.llm_branch(batch["input_ids"].to(DEVICE),
                              batch["attention_mask"].to(DEVICE))
        g     = torch.softmax(self.gate(torch.cat([h_g, h_i, h_l], dim=-1)), dim=-1)  # [B,3]
        stack = torch.stack([h_g, h_i, h_l], dim=1)                                   # [B,3,H]
        fused = (stack * g.unsqueeze(-1)).sum(dim=1)                                  # [B,H]
        logits = self.head(fused).squeeze(-1)
        aux = [self.aux_g(h_g).squeeze(-1),
               self.aux_i(h_i).squeeze(-1),
               self.aux_l(h_l).squeeze(-1)]
        return logits, aux
    def forward(self, batch):
        return self._impl(batch)[0]
    def forward_train(self, batch):
        return self._impl(batch)

print("Models defined.")

In [ ]:
# ── Cell 9: Training utilities ────────────────────────────────────────────────
import json
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def _collect(model, loader):
    model.eval()
    labels, probs = [], []
    with torch.no_grad():
        for batch in loader:
            p = torch.sigmoid(model(batch).cpu())
            probs.extend(p.tolist())
            labels.extend(batch["label"].long().tolist())
    return labels, probs

def _metrics(labels, probs, threshold=0.5):
    preds = [1 if p > threshold else 0 for p in probs]
    return {
        "f1":  f1_score(labels, preds, zero_division=0),
        "acc": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs) if len(set(labels)) > 1 else 0.0,
        "threshold": threshold,
    }

def _best_threshold(labels, probs):
    best_t, best_f1 = 0.5, -1.0
    for i in range(10, 90):
        t = i / 100
        f = f1_score(labels, [1 if p > t else 0 for p in probs], zero_division=0)
        if f > best_f1:
            best_f1, best_t = f, t
    return best_t

def evaluate(model, loader, threshold=0.5):
    labels, probs = _collect(model, loader)
    return _metrics(labels, probs, threshold)

def run_experiment(name, model, epochs=10, lr=2e-5, aux_weight=0.3, encoder_lr=None):
    print(f"\n{'='*60}")
    print(f"Training : {name}")
    print(f"{'='*60}")
    model = model.to(DEVICE)
    # Optional differential lr: fine-tune the pretrained CodeBERT encoder slowly
    # while the from-scratch graph/image/fusion params learn faster. This is the
    # standard recipe (NOT a hunch) — the aux losses below keep each branch stable,
    # which is what the earlier head=5e-4 attempt lacked.
    if encoder_lr is not None and hasattr(model, "llm_branch"):
        enc_ids     = {id(p) for p in model.llm_branch.encoder.parameters()}
        enc_params  = [p for p in model.parameters() if id(p) in enc_ids]
        rest_params = [p for p in model.parameters() if id(p) not in enc_ids]
        optimizer = torch.optim.AdamW(
            [{"params": enc_params,  "lr": encoder_lr},
             {"params": rest_params, "lr": lr}], weight_decay=1e-4)
        print(f"  encoder_lr={encoder_lr}  rest_lr={lr}")
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    criterion = nn.BCEWithLogitsLoss()
    use_amp   = DEVICE == "cuda"
    scaler    = torch.amp.GradScaler("cuda") if use_amp else None
    use_aux   = hasattr(model, "forward_train")

    # Select best epoch by val AUC (threshold-independent → stable).
    best_auc, best_state = -1.0, {k: v.cpu().clone() for k, v in model.state_dict().items()}

    def compute_loss(batch, labels):
        if use_aux:
            logits, aux = model.forward_train(batch)
            loss = criterion(logits, labels)
            for a in aux:
                loss = loss + aux_weight * criterion(a, labels)
            return loss
        return criterion(model(batch), labels)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False):
            labels = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast("cuda"):
                    loss = compute_loss(batch, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = compute_loss(batch, labels)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()

        v_labels, v_probs = _collect(model, val_loader)
        v = _metrics(v_labels, v_probs, 0.5)
        print(f"  Epoch {epoch:02d}  loss={total_loss/len(train_loader):.4f}  "
              f"val_F1@.5={v['f1']:.4f}  val_AUC={v['auc']:.4f}")
        if v["auc"] > best_auc:
            best_auc   = v["auc"]
            best_state = {k: v_.cpu().clone() for k, v_ in model.state_dict().items()}

    model.load_state_dict(best_state)
    # Tune decision threshold on validation, then report on test with that threshold.
    v_labels, v_probs = _collect(model, val_loader)
    thr = _best_threshold(v_labels, v_probs)
    test_m = evaluate(model, test_loader, threshold=thr)
    torch.save(best_state, f"{WORK_DIR}/experiments/checkpoints/{name}_best.pt")
    print(f"\n  TEST  F1={test_m['f1']:.4f}  AUC={test_m['auc']:.4f}  "
          f"Acc={test_m['acc']:.4f}  (thr={thr:.2f})")
    return test_m

os.makedirs(f"{WORK_DIR}/experiments/checkpoints", exist_ok=True)
print("Training utilities ready.")

In [ ]:
# ── Cell 10a: graph_only ──────────────────────────────────────────────────────
if "results" not in dir():
    results = {}
results["graph_only"] = run_experiment("graph_only", GraphClassifier(), epochs=15, lr=1e-4)

In [ ]:
# ── Cell 10b: image_only ──────────────────────────────────────────────────────
if "results" not in dir():
    results = {}
results["image_only"] = run_experiment("image_only", ImageClassifier(), epochs=10, lr=1e-4)

In [ ]:
# ── Cell 10c: llm_only ────────────────────────────────────────────────────────
if "results" not in dir():
    results = {}
results["llm_only"] = run_experiment("llm_only", LLMClassifier(), epochs=5, lr=2e-5)

In [ ]:
# ── Cell 10d: vulgcl (full 3-branch, gated fusion + aux losses) ──────────────
if "results" not in dir():
    results = {}
results["vulgcl"] = run_experiment("vulgcl", VulGCLModel(),
                                   epochs=10, lr=1e-4, encoder_lr=2e-5)

In [ ]:
# ── Cell 11: Results summary ──────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL RESULTS — Devign Test Set")
print("="*60)
print(f"{'Model':<20} {'F1':>8} {'AUC':>8} {'Acc':>8}")
print("-"*46)
for name, m in results.items():
    print(f"{name:<20} {m['f1']:>8.4f} {m['auc']:>8.4f} {m['acc']:>8.4f}")
print("="*60)

results_path = f"{WORK_DIR}/experiments/results.json"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {results_path}")